Instructions: Go to File > Save a copy in Drive before running. This ensures your data saves to your own Google account.

Cell 1: Environment & Folders

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import sklearn
import os
from google.colab import drive
from datetime import datetime

# 1. Install/Import yfinance
try:
    import yfinance as yf
except ImportError:
    !pip install -q yfinance
    import yfinance as yf

# 2. Mount Drive & Setup Folders
drive.mount('/content/drive')
base_dir = '/content/drive/MyDrive/MSFin_Python'
sub_folders = ['data', 'notebooks', 'output']

if not os.path.exists(base_dir):
    os.makedirs(base_dir)

for folder in sub_folders:
    path = os.path.join(base_dir, folder)
    if not os.path.exists(path):
        os.makedirs(path)

print("-" * 40)
print(f"SUCCESS: Environment and Folder Architecture Ready at {base_dir}")
print("-" * 40)

Cell 2: Data Pull & MultiIndex Fix for Close and Adj_Close

In [ ]:
# 1. Pull SPY data
ticker = "SPY"
spy_raw = yf.download(ticker, period="max", auto_adjust=False)

# 2. Fix MultiIndex Headers (Collapsing levels and removing spaces)
# This turns ('Adj Close', 'SPY') into 'Adj_Close_SPY'
spy_raw.columns = ['_'.join(filter(None, col)).strip().replace(' ', '_') for col in spy_raw.columns]

# 3. Create clean dataframe with specific columns
# Note: yfinance names these 'Close_SPY' and 'Adj_Close_SPY' now
target_cols = ['Close_SPY', 'Adj_Close_SPY']
spy_df = spy_raw[target_cols].copy()

# 4. Save Raw Data
date_str = datetime.now().strftime('%Y_%m_%d')
filename = f"{date_str}_SPY_YFIN.csv"
save_path = os.path.join(base_dir, 'data', filename)
spy_df.to_csv(save_path)

print(f"--- SUCCESS ---")
print(f"Data saved to: {save_path}")
print(f"Verified Headers: {list(spy_df.columns)}")

Cell 3: Metrics & Calculations

In [ ]:
# 1. Returns using Adj_Close_SPY
spy_df['Daily_Ret'] = spy_df['Adj_Close_SPY'].pct_change()

# 2. Monthly Returns
monthly_returns = spy_df['Adj_Close_SPY'].resample('ME').ffill().pct_change()

# 3. Annualized Information Ratio Function
def calc_info_ratio(x):
    if len(x) < 5: return np.nan
    return (x.mean() * 252) / (x.std() * np.sqrt(252))

quarterly_ir = spy_df['Daily_Ret'].resample('QE').apply(calc_info_ratio)

# 4. Summary Table
summary_df = quarterly_ir.to_frame(name='Ann_Info_Ratio')
summary_df['Year'] = summary_df.index.year
summary_df['Decade'] = (summary_df.index.year // 10) * 10
summary_df['Quarter'] = summary_df.index.quarter

print("--- CALCULATION COMPLETE ---")
print(summary_df.head())

Cell 4: Visualization (Grouped Bars)

In [ ]:
decades = sorted(summary_df['Decade'].unique())
google_colors = ['#4285F4', '#EA4335', '#FBBC05', '#34A853']

fig, axes = plt.subplots(len(decades), 1, figsize=(16, 6 * len(decades)))
if len(decades) == 1: axes = [axes]

for i, decade in enumerate(decades):
    decade_data = summary_df[summary_df['Decade'] == decade]
    pivot_df = decade_data.pivot(index='Quarter', columns='Year', values='Ann_Info_Ratio')

    pivot_df.plot(kind='bar', ax=axes[i], color=google_colors, width=0.8, edgecolor='white')

    axes[i].set_title(f"SPY Quarterly Info Ratio: {decade}s", fontsize=16, fontweight='bold')
    axes[i].set_ylabel("Annualized IR")
    axes[i].axhline(0, color='black', linewidth=1)
    axes[i].legend(title="Year", bbox_to_anchor=(1.01, 1), loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

decades = sorted(summary_df['Decade'].unique())
# Four colors mapping perfectly to the four quarters
google_colors = ['#4285F4', '#EA4335', '#FBBC05', '#34A853']

fig, axes = plt.subplots(len(decades), 1, figsize=(16, 6 * len(decades)))
if len(decades) == 1:
    axes = [axes]

for i, decade in enumerate(decades):
    decade_data = summary_df[summary_df['Decade'] == decade]

    # Swapping index and columns: Year becomes the X-axis, Quarter becomes the legend
    pivot_df = decade_data.pivot(index='Year', columns='Quarter', values='Ann_Info_Ratio')

    # Plotting grouped bars for each year
    pivot_df.plot(kind='bar', ax=axes[i], color=google_colors, width=0.8, edgecolor='white')

    axes[i].set_title(f"SPY Quarterly Info Ratio: {decade}s", fontsize=16, fontweight='bold')
    axes[i].set_ylabel("Annualized IR")
    axes[i].set_xlabel("Year")
    axes[i].axhline(0, color='black', linewidth=1)

    # Updating legend title to "Quarter"
    axes[i].legend(title="Quarter", bbox_to_anchor=(1.01, 1), loc='upper left')

    # Rotating x-labels for better readability
    axes[i].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('spy_quarterly_info_ratio.png')

Cell 5: Report Export

In [ ]:
output_date = datetime.now().strftime('%Y_%m_%d')
summary_filename = f"{output_date}_SPY_Performance_Summary.csv"
output_path = os.path.join(base_dir, 'output', summary_filename)

summary_df.to_csv(output_path)

print("-" * 40)
print(f"REPORT EXPORTED: {output_path}")
print("-" * 40)